# Step 11: Performance Profiling & Bottleneck Diagnosis

## Learning Objectives
1. Classify the 5 types of performance problems
2. Systematic diagnosis using the Spark UI
3. Event Log analysis
4. Reproduce and resolve real-world bottleneck scenarios
5. Advanced AQE (Adaptive Query Execution)
6. Profiling checklist

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import random
import time
import os

spark = SparkSession.builder \
    .appName('Step11-Performance-Profiling') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.executor.cores', '1') \
    .config('spark.sql.shuffle.partitions', '20') \
    .config('spark.sql.adaptive.enabled', 'false') \
    .config('spark.sql.warehouse.dir', '/home/jovyan/data/warehouse') \
    .getOrCreate()

sc = spark.sparkContext

print(f'Spark version: {spark.version}')
print(f'✅ Spark UI: http://localhost:4040')


Spark version: 3.5.0
✅ Spark UI: http://localhost:4040


---
## 1. The 5 Types of Performance Problems

```
┌─────────────────────────────────────────────────────────────┐
│           Spark Performance Problem Classification          │
├──────────────────┬──────────────────────────────────────────┤
│ Type             │ Symptoms & Causes                        │
├──────────────────┼──────────────────────────────────────────┤
│ 1. Data Skew     │ Most Tasks fast, 1~2 extremely slow      │
│                  │ → Data concentrated on specific key      │
├──────────────────┼──────────────────────────────────────────┤
│ 2. Excess Shuffle│ Slow Stage transitions, network bottleneck│
│                  │ → Unnecessary Shuffle, poor partitioning  │
├──────────────────┼──────────────────────────────────────────┤
│ 3. Spill         │ Disk I/O spikes, Task duration grows     │
│                  │ → Partition larger than available memory  │
├──────────────────┼──────────────────────────────────────────┤
│ 4. Excess GC     │ GC Time > 10% of Task Time              │
│                  │ → Over-caching, large heap, many objects  │
├──────────────────┼──────────────────────────────────────────┤
│ 5. Serialization │ Python UDF, collect, large broadcast     │
│                  │ → JVM ↔ Python round-trips               │
└──────────────────┴──────────────────────────────────────────┘
```

---
## 2. Diagnostic Tool: Systematic Spark UI Usage

```
Problem detected!
    │
    ▼
① Jobs tab: which Job is slow?
    │
    ▼
② Stages tab: which Stage is slow?
    │          how large is the Shuffle Read/Write?
    ▼
③ Stage detail: inspect Task time distribution
    │            max >> median? → Skew
    │            Spill (Memory/Disk)?
    │            GC Time?
    ▼
④ SQL tab: Exchange location in DAG, check row counts
    │
    ▼
⑤ Executors tab: GC Time, memory usage
    │
    ▼
Diagnosis complete → apply fix
```

In [2]:
# Diagnostic utility functions

def profile(name, func):
    """Measure query execution time + Spark UI guidance"""
    print(f'\n{"=" * 60}')
    print(f'  {name}')
    print(f'{"=" * 60}')
    
    start = time.time()
    result = func()
    elapsed = time.time() - start
    
    print(f'  Elapsed: {elapsed:.3f}s')
    print(f'  → Check the latest Job in Spark UI: http://localhost:4040/jobs/')
    return elapsed, result

def compare(name_a, func_a, name_b, func_b):
    """Compare two approaches"""
    t_a, _ = profile(name_a, func_a)
    t_b, _ = profile(name_b, func_b)
    
    winner = name_a if t_a < t_b else name_b
    ratio = max(t_a, t_b) / min(t_a, t_b) if min(t_a, t_b) > 0 else 0
    print(f'\n  📊 {winner} wins! ({ratio:.1f}x faster)')
    return t_a, t_b

---
## 3. Scenario 1: Diagnosing and Resolving Data Skew

In [3]:
# Generate skewed data
random.seed(42)
n = 1_000_000

skew_data = []
for i in range(n):
    if random.random() < 0.7:
        key = 'hot_key'  # 70% on a single key
    else:
        key = f'key_{random.randint(1, 100)}'
    skew_data.append((i, key, random.randint(1, 1000)))

skew_df = spark.createDataFrame(skew_data, ['id', 'key', 'value'])
skew_df.cache().count()

# Check skew distribution
print('=== Key distribution (top 5) ===')
skew_df.groupBy('key').count().orderBy(F.col('count').desc()).show(5)

print('⚠️ 700K rows concentrated on hot_key → one Task is extremely slow')
print('   Check max >> median in Spark UI > Stages > Summary Metrics')

=== Key distribution (top 5) ===
+-------+------+
|    key| count|
+-------+------+
|hot_key|699863|
| key_79|  3121|
| key_18|  3111|
| key_68|  3100|
| key_80|  3099|
+-------+------+
only showing top 5 rows

⚠️ 700K rows concentrated on hot_key → one Task is extremely slow
   Check max >> median in Spark UI > Stages > Summary Metrics


In [4]:
# Fix 1: Salting
num_salts = 20

# Deterministic evidence: size of the hottest reduce partition (the straggler), before vs after salting.
def hottest_partition(df, key, n=20):
    counts = [r['c'] for r in (df.repartition(n, key)
              .withColumn('p', F.spark_partition_id())
              .groupBy('p').agg(F.count('*').alias('c')).collect())]
    return max(counts)

before_hot = hottest_partition(skew_df, 'key')
salted_df = skew_df.withColumn('salted_key',
    F.concat('key', F.lit('_'), (F.rand() * num_salts).cast('int').cast('string')))
after_hot = hottest_partition(salted_df, 'salted_key')
print(f'Hottest partition BEFORE salting (by key)        : {before_hot:,} rows')
print(f'Hottest partition AFTER  salting (by salted_key) : {after_hot:,} rows  -> ~{before_hot/after_hot:.0f}x smaller\n')

def skew_naive():
    return skew_df.groupBy('key').agg(F.sum('value').alias('total')).collect()

def skew_salted():
    salted = skew_df.withColumn('salt', (F.rand() * num_salts).cast('int'))
    partial = salted.groupBy('key', 'salt').agg(F.sum('value').alias('partial_sum'))
    return partial.groupBy('key').agg(F.sum('partial_sum').alias('total')).collect()

compare('Plain groupBy', skew_naive, 'Salting groupBy', skew_salted)

print('''
⚠️ For groupBy(sum/count/avg) the plain version usually WINS — and that is the lesson here.
   These aggregates are combinable, so Spark does map-side partial aggregation BEFORE the
   shuffle: the hot key never reaches a single reduce task in full, so skew does not hurt and
   salting only adds a second aggregation pass. The hottest-partition numbers above are the
   real signal that the data IS skewed; salting matters when that full hot partition reaches
   one task — i.e. a shuffled JOIN (next cell) or a non-combinable aggregation.
''')

Hottest partition BEFORE salting (by key)        : 708,753 rows
Hottest partition AFTER  salting (by salted_key) : 120,611 rows  -> ~6x smaller


  Plain groupBy
  Elapsed: 0.337s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  Salting groupBy
  Elapsed: 0.551s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  📊 Plain groupBy wins! (1.6x faster)

⚠️ For groupBy(sum/count/avg) the plain version usually WINS — and that is the lesson here.
   These aggregates are combinable, so Spark does map-side partial aggregation BEFORE the
   shuffle: the hot key never reaches a single reduce task in full, so skew does not hurt and
   salting only adds a second aggregation pass. The hottest-partition numbers above are the
   real signal that the data IS skewed; salting matters when that full hot partition reaches
   one task — i.e. a shuffled JOIN (next cell) or a non-combinable aggregation.



In [5]:
# Fix 2: AQE Skew Join
lookup = spark.createDataFrame(
    [('hot_key', 'Hot')] + [(f'key_{i}', f'Desc_{i}') for i in range(1, 101)],
    ['key', 'description']
)

def join_no_aqe():
    spark.conf.set('spark.sql.adaptive.enabled', 'false')
    return skew_df.join(lookup, 'key').count()

def join_aqe():
    spark.conf.set('spark.sql.adaptive.enabled', 'true')
    spark.conf.set('spark.sql.adaptive.skewJoin.enabled', 'true')
    spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionFactor', '5')
    spark.conf.set('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes', '64m')
    return skew_df.join(lookup, 'key').count()

compare('Join (AQE OFF)', join_no_aqe, 'Join (AQE ON)', join_aqe)
spark.conf.set('spark.sql.adaptive.enabled', 'false')


  Join (AQE OFF)
  Elapsed: 0.774s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  Join (AQE ON)
  Elapsed: 0.374s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  📊 Join (AQE ON) wins! (2.1x faster)


---
## 4. Scenario 2: Diagnosing and Reducing Excessive Shuffle

In [6]:
# Large table + small table join
random.seed(42)
big = spark.range(1_000_000).withColumn('dept', (F.col('id') % 50).cast('string')) \
    .withColumn('salary', F.rand() * 100000)
big.cache().count()

small = spark.createDataFrame(
    [(str(i), f'Department_{i}', random.choice(['US','EU','APAC'])) for i in range(50)],
    ['dept', 'dept_name', 'region']
)

# ❌ SortMergeJoin (Shuffle on both sides)
def shuffle_join():
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
    return big.join(small, 'dept').groupBy('region').agg(F.avg('salary')).collect()

# ✅ BroadcastHashJoin (eliminates Shuffle)
def broadcast_join():
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10485760')
    return big.join(F.broadcast(small), 'dept').groupBy('region').agg(F.avg('salary')).collect()

compare('SortMergeJoin (Shuffle)', shuffle_join, 'BroadcastHashJoin (No Shuffle)', broadcast_join)

print('''
🔍 Compare in Spark UI:
   SortMergeJoin → 2 Exchange (Shuffle) nodes in Stages
   BroadcastHashJoin → 0 (or 1) Exchange node
   Check the difference in Shuffle Read/Write bytes.
''')


  SortMergeJoin (Shuffle)
  Elapsed: 0.886s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  BroadcastHashJoin (No Shuffle)
  Elapsed: 0.462s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  📊 BroadcastHashJoin (No Shuffle) wins! (1.9x faster)

🔍 Compare in Spark UI:
   SortMergeJoin → 2 Exchange (Shuffle) nodes in Stages
   BroadcastHashJoin → 0 (or 1) Exchange node
   Check the difference in Shuffle Read/Write bytes.



In [7]:
# Eliminate unnecessary Shuffle: aggregate before joining

# ❌ Join then aggregate (Shuffle on large data)
def join_then_agg():
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
    return big.join(small, 'dept') \
        .groupBy('region') \
        .agg(F.avg('salary').alias('avg_sal'), F.count('*').alias('cnt')) \
        .collect()

# ✅ Aggregate then join (Shuffle on small results)
def agg_then_join():
    spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
    dept_summary = big.groupBy('dept').agg(
        F.avg('salary').alias('avg_sal'),
        F.count('*').alias('cnt')
    )
    return dept_summary.join(small, 'dept') \
        .groupBy('region') \
        .agg(
            (F.sum(F.col('avg_sal') * F.col('cnt')) / F.sum('cnt')).alias('weighted_avg'),
            F.sum('cnt').alias('total_cnt')
        ) \
        .collect()

compare('Join→Aggregate (large Shuffle)', join_then_agg, 'Aggregate→Join (small Shuffle)', agg_then_join)

spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10485760')


  Join→Aggregate (large Shuffle)
  Elapsed: 0.680s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  Aggregate→Join (small Shuffle)
  Elapsed: 0.509s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  📊 Aggregate→Join (small Shuffle) wins! (1.3x faster)


---
## 5. Scenario 3: Diagnosing and Resolving Spill

In [8]:
import json, urllib.request

# Trigger Spill: sort ~1GB of data through very few partitions.
# NOTE: we use .write.format('noop') (a sink that consumes every row) — NOT .count() —
# because orderBy().count() lets Catalyst prune the Sort, so no spill would ever occur.
wide_df = spark.range(2_000_000) \
    .withColumn('payload', F.expr("repeat('x', 480)")) \
    .withColumn('value', F.rand())

def total_spill_mb():
    url = f'{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/stages'
    stages = json.load(urllib.request.urlopen(url))
    mem = sum(s.get('memoryBytesSpilled', 0) for s in stages) / 1024 / 1024
    disk = sum(s.get('diskBytesSpilled', 0) for s in stages) / 1024 / 1024
    return mem, disk

def sorted_run(nparts):
    spark.conf.set('spark.sql.shuffle.partitions', str(nparts))
    m0, d0 = total_spill_mb()                     # cumulative spill BEFORE this run
    t = time.time()
    wide_df.orderBy('value').write.format('noop').mode('overwrite').save()  # forces the sort
    elapsed = time.time() - t
    m1, d1 = total_spill_mb()                     # cumulative spill AFTER
    return elapsed, m1 - m0, d1 - d0              # report only THIS run's delta

t2,  m2,  d2  = sorted_run(2)     # ~1GB into 2 reduce partitions -> spills
t20, m20, d20 = sorted_run(20)    # small per-partition -> no spill
spark.conf.set('spark.sql.shuffle.partitions', '20')

print(f'  2 partitions  -> {t2:5.2f}s | spill(memory) {m2:6.0f} MB | spill(disk) {d2:5.0f} MB')
print(f' 20 partitions  -> {t20:5.2f}s | spill(memory) {m20:6.0f} MB | spill(disk) {d20:5.0f} MB')

print('''
🔍 Spark UI > Stages > Stage detail > Summary Metrics: 'Spill (memory)' / 'Spill (disk)'.
   'Spill (memory)' = uncompressed size evicted from execution memory; 'Spill (disk)' =
   compressed size actually written. With 2 partitions ~1GB cannot fit one task's execution
   memory, so it spills; with 20 partitions each task sorts a few MB and nothing spills.

   If Spill appears:
   1. Increase shuffle.partitions (reduce per-partition size)
   2. Increase executor.memory
   3. Drop unnecessary large columns (like payload)
   (And to MEASURE spill, consume the rows with .write/.agg — .count() prunes the sort.)
''')

  2 partitions  ->  1.13s | spill(memory)    736 MB | spill(disk)    39 MB
 20 partitions  ->  0.48s | spill(memory)      0 MB | spill(disk)     0 MB

🔍 Spark UI > Stages > Stage detail > Summary Metrics: 'Spill (memory)' / 'Spill (disk)'.
   'Spill (memory)' = uncompressed size evicted from execution memory; 'Spill (disk)' =
   compressed size actually written. With 2 partitions ~1GB cannot fit one task's execution
   memory, so it spills; with 20 partitions each task sorts a few MB and nothing spills.

   If Spill appears:
   1. Increase shuffle.partitions (reduce per-partition size)
   2. Increase executor.memory
   3. Drop unnecessary large columns (like payload)
   (And to MEASURE spill, consume the rows with .write/.agg — .count() prunes the sort.)



---
## 6. Scenario 4: Serialization Bottleneck (Python UDF vs Built-in Functions)

In [9]:
from pyspark.sql.functions import udf, pandas_udf
import pandas as pd
import numpy as np
import math

perf_df = spark.range(1_000_000).withColumn('value', F.rand() * 1000)
perf_df.cache().count()

# 1. Python UDF
@udf('double')
def python_transform(v):
    if v is None: return None
    return math.log(v + 1) * math.sqrt(v)

# 2. Pandas UDF
@pandas_udf('double')
def pandas_transform(v: pd.Series) -> pd.Series:
    return np.log(v + 1) * np.sqrt(v)

# IMPORTANT: aggregate the result (.agg(F.sum)) to FORCE the UDF to run on every row.
# .count() does not need the computed column, so Catalyst prunes the UDF and it never executes
# — making all three look identical. Summing the result gives the real serialization cost.
def builtin_transform():
    return perf_df.select((F.log(F.col('value') + 1) * F.sqrt('value')).alias('r')).agg(F.sum('r')).collect()

def python_udf_run():
    return perf_df.select(python_transform('value').alias('r')).agg(F.sum('r')).collect()

def pandas_udf_run():
    return perf_df.select(pandas_transform('value').alias('r')).agg(F.sum('r')).collect()

# Warm up each path once (the first Pandas-UDF call pays one-time Arrow / pandas worker init;
# without a warm-up that cost is wrongly charged to Pandas UDF and can make it look slower).
builtin_transform(); pandas_udf_run(); python_udf_run()

t1, _ = profile('Built-in function', builtin_transform)
t2, _ = profile('Pandas UDF', pandas_udf_run)
t3, _ = profile('Python UDF', python_udf_run)

print(f'''
=== Performance Summary (1M rows, result materialized, warmed up) ===
  Built-in:   {t1:.3f}s (baseline)
  Pandas UDF: {t2:.3f}s ({t2/t1:.1f}x)
  Python UDF: {t3:.3f}s ({t3/t1:.1f}x)

  Why Python UDF is slow:
  1. JVM -> Python serialization (pickle) per row
  2. Python GIL contention
  3. Codegen broken -> falls back to Volcano model
''')

perf_df.unpersist()


  Built-in function
  Elapsed: 0.053s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  Pandas UDF
  Elapsed: 0.137s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

  Python UDF
  Elapsed: 0.350s
  → Check the latest Job in Spark UI: http://localhost:4040/jobs/

=== Performance Summary (1M rows, result materialized, warmed up) ===
  Built-in:   0.053s (baseline)
  Pandas UDF: 0.137s (2.6x)
  Python UDF: 0.350s (6.6x)

  Why Python UDF is slow:
  1. JVM -> Python serialization (pickle) per row
  2. Python GIL contention
  3. Codegen broken -> falls back to Volcano model



DataFrame[id: bigint, value: double]

---
## 7. Advanced AQE (Adaptive Query Execution)

In [10]:
print('''
=== AQE's 3 Optimizations ===

1. Coalescing Shuffle Partitions
   ┌──┐┌──┐┌──┐┌──┐┌──┐┌──┐┌──┐┌──┐  (200 partitions, mostly tiny)
   │  ││  ││  ││  ││  ││  ││  ││  │
   └──┘└──┘└──┘└──┘└──┘└──┘└──┘└──┘
         ↓ AQE: merge small partitions
   ┌────────┐┌────────┐┌────────┐      (merged into 3)
   │        ││        ││        │
   └────────┘└────────┘└────────┘

2. Switching Join Strategy
   Initial plan: SortMergeJoin (inaccurate statistics)
         ↓ check actual sizes after Shuffle
   Switch to: BroadcastHashJoin (one side confirmed small)

3. Skew Join Optimization
   ┌──┐┌──┐┌──────────────┐┌──┐  (partition 3 is skewed)
   │  ││  ││              ││  │
   └──┘└──┘└──────────────┘└──┘
         ↓ AQE: split skewed partition
   ┌──┐┌──┐┌────┐┌────┐┌────┐┌──┐
   │  ││  ││    ││    ││    ││  │
   └──┘└──┘└────┘└────┘└────┘└──┘
''')


=== AQE's 3 Optimizations ===

1. Coalescing Shuffle Partitions
   ┌──┐┌──┐┌──┐┌──┐┌──┐┌──┐┌──┐┌──┐  (200 partitions, mostly tiny)
   │  ││  ││  ││  ││  ││  ││  ││  │
   └──┘└──┘└──┘└──┘└──┘└──┘└──┘└──┘
         ↓ AQE: merge small partitions
   ┌────────┐┌────────┐┌────────┐      (merged into 3)
   │        ││        ││        │
   └────────┘└────────┘└────────┘

2. Switching Join Strategy
   Initial plan: SortMergeJoin (inaccurate statistics)
         ↓ check actual sizes after Shuffle
   Switch to: BroadcastHashJoin (one side confirmed small)

3. Skew Join Optimization
   ┌──┐┌──┐┌──────────────┐┌──┐  (partition 3 is skewed)
   │  ││  ││              ││  │
   └──┘└──┘└──────────────┘└──┘
         ↓ AQE: split skewed partition
   ┌──┐┌──┐┌────┐┌────┐┌────┐┌──┐
   │  ││  ││    ││    ││    ││  │
   └──┘└──┘└────┘└────┘└────┘└──┘



In [11]:
# AQE Coalescing experiment
random.seed(42)
test_data = [(i, random.choice(['A','B','C']), random.randint(1,100)) for i in range(100_000)]
test_df = spark.createDataFrame(test_data, ['id', 'grp', 'val'])

# AQE OFF: keep 200 partitions as-is
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '200')

start = time.time()
r1 = test_df.groupBy('grp').agg(F.sum('val'))
r1.collect()
aqe_off_time = time.time() - start

# AQE ON: auto-coalesce partitions
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')

start = time.time()
r2 = test_df.groupBy('grp').agg(F.sum('val'))
r2.collect()
aqe_on_time = time.time() - start

print(f'AQE OFF (200 partitions): {aqe_off_time:.3f}s')
print(f'AQE ON  (auto-coalesced): {aqe_on_time:.3f}s')

print('''
💡 If data is small but 200 partitions are used:
   - Almost no data per partition
   - Task overhead dominates
   → AQE automatically coalesces to an appropriate count
''')

spark.conf.set('spark.sql.shuffle.partitions', '20')

AQE OFF (200 partitions): 0.579s
AQE ON  (auto-coalesced): 0.141s

💡 If data is small but 200 partitions are used:
   - Almost no data per partition
   - Task overhead dominates
   → AQE automatically coalesces to an appropriate count



In [12]:
# Full list of AQE-related settings
aqe_configs = [
    ('spark.sql.adaptive.enabled', 'Master switch'),
    ('spark.sql.adaptive.coalescePartitions.enabled', 'Coalesce partitions'),
    ('spark.sql.adaptive.coalescePartitions.minPartitionSize', 'Min partition size when coalescing'),
    ('spark.sql.adaptive.coalescePartitions.initialPartitionNum', 'Initial partition count'),
    ('spark.sql.adaptive.skewJoin.enabled', 'Skew Join optimization'),
    ('spark.sql.adaptive.skewJoin.skewedPartitionFactor', 'Skew detection multiplier'),
    ('spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes', 'Minimum size to consider skewed'),
    ('spark.sql.adaptive.localShuffleReader.enabled', 'Local Shuffle reader'),
]

print('=== AQE Settings ===')
for key, desc in aqe_configs:
    try:
        val = spark.conf.get(key)
    except Exception:
        val = '(default)'
    print(f'  {str(val):>10}  {key}')
    print(f'             {desc}')
    print()

print('''
💡 Recommended production settings for Spark 3.x:
   spark.sql.adaptive.enabled = true
   spark.sql.adaptive.coalescePartitions.enabled = true
   spark.sql.adaptive.skewJoin.enabled = true
   spark.sql.shuffle.partitions = 200 (AQE will auto-adjust)
''')

=== AQE Settings ===
        true  spark.sql.adaptive.enabled
             Master switch

        true  spark.sql.adaptive.coalescePartitions.enabled
             Coalesce partitions

    1048576b  spark.sql.adaptive.coalescePartitions.minPartitionSize
             Min partition size when coalescing

        None  spark.sql.adaptive.coalescePartitions.initialPartitionNum
             Initial partition count

        true  spark.sql.adaptive.skewJoin.enabled
             Skew Join optimization

           5  spark.sql.adaptive.skewJoin.skewedPartitionFactor
             Skew detection multiplier

         64m  spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes
             Minimum size to consider skewed

        true  spark.sql.adaptive.localShuffleReader.enabled
             Local Shuffle reader


💡 Recommended production settings for Spark 3.x:
   spark.sql.adaptive.enabled = true
   spark.sql.adaptive.coalescePartitions.enabled = true
   spark.sql.adaptive.skewJoin.enabled 

---
## 8. Event Log Analysis

In [13]:
print('''
=== Using the Event Log ===

Configuration:
  spark.eventLog.enabled = true
  spark.eventLog.dir = hdfs:///spark-events  (or a local path)

What is recorded in the Event Log:
  - Job/Stage/Task start and end times
  - Shuffle Read/Write bytes
  - Spill bytes
  - GC time
  - Execution plan
  - Environment configuration

Analyze with Spark History Server:
  $ ./sbin/start-history-server.sh
  → http://localhost:18080
  → View the Spark UI for completed applications as-is

Programmatic analysis:
  Event Log is in JSON format and can be read directly with Spark:
''')

# Check Event Log files
event_dir = '/home/jovyan/data/spark-events'
if os.path.exists(event_dir):
    files = os.listdir(event_dir)
    print(f'Event Log files: {len(files)}')
    for f in files[:3]:
        size_kb = os.path.getsize(os.path.join(event_dir, f)) / 1024
        print(f'  {f}: {size_kb:.1f} KB')


=== Using the Event Log ===

Configuration:
  spark.eventLog.enabled = true
  spark.eventLog.dir = hdfs:///spark-events  (or a local path)

What is recorded in the Event Log:
  - Job/Stage/Task start and end times
  - Shuffle Read/Write bytes
  - Spill bytes
  - GC time
  - Execution plan
  - Environment configuration

Analyze with Spark History Server:
  $ ./sbin/start-history-server.sh
  → http://localhost:18080
  → View the Spark UI for completed applications as-is

Programmatic analysis:
  Event Log is in JSON format and can be read directly with Spark:



---
## 9. Comprehensive Profiling Checklist

In [14]:
print('''
╔══════════════════════════════════════════════════════════════════╗
║          Spark Performance Profiling Checklist                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  🔍 Step 1: Identify the problem                               ║
║  □ Which Job/Stage is slow? (Jobs/Stages tab)                  ║
║  □ Everything slow vs only specific Tasks?                     ║
║  □ CPU-bound vs I/O-bound?                                     ║
║                                                                ║
║  📊 Step 2: Check the data                                     ║
║  □ Data Skew? (Task time: max >> median)                       ║
║  □ Appropriate data size? (target 100~200MB per partition)     ║
║  □ Are unnecessary columns being passed through?               ║
║                                                                ║
║  🔄 Step 3: Check Shuffle                                      ║
║  □ Check Shuffle Read/Write sizes                              ║
║  □ Is the small table being broadcast?                         ║
║  □ Any unnecessary Shuffle? (check explain)                    ║
║  □ Are filters/aggregations applied before the Join?           ║
║                                                                ║
║  💾 Step 4: Check memory                                       ║
║  □ Spill (Memory/Disk) occurring? → increase partition count   ║
║  □ GC Time > 10%? → over-caching / insufficient memory        ║
║  □ OOM? → increase executor.memory / reduce data              ║
║                                                                ║
║  ⚙️ Step 5: Check code                                        ║
║  □ Python UDF → replace with built-in or Pandas UDF           ║
║  □ Minimize collect()/toPandas()                               ║
║  □ Validate execution plan with explain()                      ║
║  □ Is WholeStageCodegen still intact?                          ║
║                                                                ║
║  🎛️ Step 6: Tune settings                                     ║
║  □ Enable AQE (Spark 3.x)                                      ║
║  □ Is spark.sql.shuffle.partitions appropriate?                ║
║  □ Executor memory/core balance                                ║
║  □ Check autoBroadcastJoinThreshold                            ║
║                                                                ║
╚══════════════════════════════════════════════════════════════════╝
''')


╔══════════════════════════════════════════════════════════════════╗
║          Spark Performance Profiling Checklist                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                ║
║  🔍 Step 1: Identify the problem                               ║
║  □ Which Job/Stage is slow? (Jobs/Stages tab)                  ║
║  □ Everything slow vs only specific Tasks?                     ║
║  □ CPU-bound vs I/O-bound?                                     ║
║                                                                ║
║  📊 Step 2: Check the data                                     ║
║  □ Data Skew? (Task time: max >> median)                       ║
║  □ Appropriate data size? (target 100~200MB per partition)     ║
║  □ Are unnecessary columns being passed through?               ║
║                                                                ║
║  🔄 Step 3: Check Shuffle                                

In [15]:
print('''
=== Common Performance Problems → Quick Fix Reference ===

┌────────────────────────┬─────────────────────────────────────┐
│ Symptom                │ Fix                                 │
├────────────────────────┼─────────────────────────────────────┤
│ Only 1~2 Tasks slow    │ Data Skew → Salting / AQE           │
│ Shuffle is very large  │ Broadcast Join / aggregate first    │
│ Spill occurring        │ Increase partitions / add memory    │
│ GC Time > 10%          │ SER cache / more memory / G1 GC    │
│ Python UDF slow        │ Built-in functions / Pandas UDF     │
│ OOM (Driver)           │ Remove collect / increase driver.mem│
│ OOM (Executor)         │ More partitions / more executor.mem │
│ Too many partitions    │ AQE coalesce / reduce partition cnt │
│ Slow overall           │ Review execution plan / check Codegen│
│ Slow file reads        │ Use Parquet / partitioning / pushdown│
└────────────────────────┴─────────────────────────────────────┘
''')


=== Common Performance Problems → Quick Fix Reference ===

┌────────────────────────┬─────────────────────────────────────┐
│ Symptom                │ Fix                                 │
├────────────────────────┼─────────────────────────────────────┤
│ Only 1~2 Tasks slow    │ Data Skew → Salting / AQE           │
│ Shuffle is very large  │ Broadcast Join / aggregate first    │
│ Spill occurring        │ Increase partitions / add memory    │
│ GC Time > 10%          │ SER cache / more memory / G1 GC    │
│ Python UDF slow        │ Built-in functions / Pandas UDF     │
│ OOM (Driver)           │ Remove collect / increase driver.mem│
│ OOM (Executor)         │ More partitions / more executor.mem │
│ Too many partitions    │ AQE coalesce / reduce partition cnt │
│ Slow overall           │ Review execution plan / check Codegen│
│ Slow file reads        │ Use Parquet / partitioning / pushdown│
└────────────────────────┴─────────────────────────────────────┘



---
## 📝 Advanced Course Key Takeaways

| Step | Topic | Key Learning |
|------|-------|--------------|
| 8 | Pandas UDF & Arrow | Accelerate UDFs with vectorization, model inference with Iterator |
| 9 | Spark Internal Architecture | Job→Stage→Task, DAG Scheduler, data locality |
| 10 | Tungsten & Codegen | UnsafeRow, cache-friendly computation, code generation |
| 11 | Performance Profiling | 5 bottleneck types, systematic diagnosis, AQE usage |

### Recommended Production Settings (Spark 3.x)
```
spark.sql.adaptive.enabled = true
spark.sql.adaptive.coalescePartitions.enabled = true
spark.sql.adaptive.skewJoin.enabled = true
spark.sql.shuffle.partitions = 200
spark.sql.autoBroadcastJoinThreshold = 10485760
spark.sql.codegen.wholeStage = true
spark.sql.execution.arrow.pyspark.enabled = true
spark.executor.memory = 4g~8g
spark.executor.cores = 4~5
spark.serializer = org.apache.spark.serializer.KryoSerializer
```

### 🎉 Advanced Course Complete!
Foundations (Step 1~7) + Advanced (Step 8~11) — from Spark usage to internals, all covered.

In [16]:
spark.stop()
print('SparkSession stopped')
print('\n🎉 All 11 steps complete!')

SparkSession stopped

🎉 All 11 steps complete!
